# Installed Base Reliability and Service Cost Forecasting
## Part 1: Data Foundation and Failure Behavior

### Business context
Companies like KLA sell high value process control and inspection tools to semiconductor fabs and then service that installed base for years, mostly under multi year service contracts. A service contract is a fixed price promise made against an uncertain future cost: the company collects a known amount and absorbs whatever failures actually happen. Forecasting that cost well decides contract pricing, spare parts stocking, and financial reserves.

### The question this project answers
> Given an installed base of machines of different models and ages, how often does each component fail, what drives those failures, and what will servicing the fleet cost next quarter, including the uncertainty around that number?

### Dataset
The Microsoft Azure PdM (Predictive Maintenance) dataset: 100 machines, 4 component types, one year of telemetry, error logs, maintenance records, and failure records. It is a public proxy for a tool fleet: machines play the role of tools, `model` plays the role of tool platform or generation, and `comp1` to `comp4` play the role of serviceable modules.

### Project roadmap
| Part | Notebook | Output |
|---|---|---|
| **1** | **Data foundation and failure behavior (this notebook)** | **Clean component lifetime table with censoring** |
| 2 | Survival modeling | Kaplan Meier curves, Weibull fits, Cox PH (Proportional Hazards) drivers |
| 3 | Fleet cost forecast | Quarterly failure and cost forecast, backtested |
| 4 | New generation forecasting | Hierarchical Bayesian model for sparse data |

### What this notebook produces
One table, `component_lifetimes.csv`, with one row per component lifetime: how long it lasted, and whether it ended in a **failure** or was **censored** (still working when we stopped watching it). Every model in Parts 2 to 4 is built on this table, so getting it right is the most important step of the project.

## 1. Setup
Imports, file paths, and plotting defaults. The dataset is five CSV (Comma Separated Values) files.

**Download:** get the files from Kaggle ([Microsoft Azure Predictive Maintenance](https://www.kaggle.com/datasets/arnabbiswas1/microsoft-azure-predictive-maintenance)) and place them in `data/raw/`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

DATA_RAW = Path("data/raw")
DATA_PROCESSED = Path("data/processed")
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

COMPONENTS = ["comp1", "comp2", "comp3", "comp4"]

pd.set_option("display.max_columns", 20)
plt.rcParams.update({"figure.figsize": (9, 4), "axes.grid": True, "grid.alpha": 0.3})

## 2. Load the data
Each file is a different view of the same fleet:

| File | Grain (one row per) | Role in the service analogy |
|---|---|---|
| `PdM_machines` | machine | installed base register: platform and age |
| `PdM_telemetry` | machine per hour | sensor readings (volt, rotate, pressure, vibration) |
| `PdM_errors` | error event | non fatal alarms the tool raised |
| `PdM_maint` | component replacement | field service visits that swapped a part |
| `PdM_failures` | failure event | replacements that happened *because* the part failed |

The key relationship: **every failure is also a maintenance record**, but not every maintenance record is a failure. Some replacements are preventive.

In [ ]:
def load(name: str, parse_dates: bool = True) -> pd.DataFrame:
    df = pd.read_csv(DATA_RAW / f"PdM_{name}.csv")
    if parse_dates and "datetime" in df.columns:
        df["datetime"] = pd.to_datetime(df["datetime"])
    return df

machines = load("machines", parse_dates=False)
telemetry = load("telemetry")
errors = load("errors")
maint = load("maint")
failures = load("failures")

for name, df in [("machines", machines), ("telemetry", telemetry), ("errors", errors),
                 ("maint", maint), ("failures", failures)]:
    print(f"{name:<10} {df.shape[0]:>8,} rows  columns: {list(df.columns)}")

## 3. Data quality profile
Before modeling, confirm three things for every table: the date coverage, missing values, and duplicate keys. A duplicated failure record would silently inflate the failure rate, which directly inflates the cost forecast.

In [ ]:
def profile(df: pd.DataFrame, key: list[str]) -> dict:
    out = {"rows": len(df), "nulls": int(df.isna().sum().sum()),
           "duplicate_keys": int(df.duplicated(subset=key).sum())}
    if "datetime" in df.columns:
        out["start"] = df["datetime"].min()
        out["end"] = df["datetime"].max()
    return out

quality = pd.DataFrame({
    "machines": profile(machines, ["machineID"]),
    "telemetry": profile(telemetry, ["machineID", "datetime"]),
    "errors": profile(errors, ["machineID", "datetime", "errorID"]),
    "maint": profile(maint, ["machineID", "datetime", "comp"]),
    "failures": profile(failures, ["machineID", "datetime", "failure"]),
}).T
quality

**Notice the date ranges.** Maintenance records start in 2014, but failures and telemetry only cover 2015. That means a replacement recorded in 2014 cannot be classified as failure or preventive, because we cannot see failures in that year. We will define an **observation window** from telemetry and only classify lifetimes that end inside it. This is the kind of incomplete data issue the real service data will have too.

In [ ]:
OBS_START = telemetry["datetime"].min()
OBS_END = telemetry["datetime"].max()
print(f"Observation window: {OBS_START}  to  {OBS_END}  ({(OBS_END - OBS_START).days} days)")

# Remove exact duplicate rows if any exist
maint = maint.drop_duplicates()
failures = failures.drop_duplicates()

## 4. The installed base
In service forecasting the installed base is the denominator of everything: 30 failures means something very different on 50 tools than on 500. Here we look at how the fleet is distributed across models (platforms) and ages.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))

machines["model"].value_counts().sort_index().plot.bar(ax=axes[0], color="#534AB7")
axes[0].set(title="Installed base by model", xlabel="Model", ylabel="Machines")

for model, grp in machines.groupby("model"):
    axes[1].hist(grp["age"], bins=range(0, 22), alpha=0.5, label=model)
axes[1].set(title="Machine age distribution", xlabel="Age (years)", ylabel="Machines")
axes[1].legend()
plt.tight_layout()
plt.show()

machines.groupby("model")["age"].describe()[["count", "mean", "min", "max"]]

## 5. Failure behavior
Three views that a service planner would ask for first:
1. Which components fail most?
2. Does that differ by model?
3. Are failures spread evenly across machines, or concentrated in a few "problem tools"?

In [ ]:
fail_by_comp_model = pd.crosstab(failures["failure"], failures.merge(machines, on="machineID")["model"])
display(fail_by_comp_model)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
failures["failure"].value_counts().reindex(COMPONENTS).plot.bar(ax=axes[0], color="#D85A30")
axes[0].set(title="Failures by component (observation window)", xlabel="", ylabel="Failures")

per_machine = failures.groupby("machineID").size().reindex(machines["machineID"], fill_value=0)
axes[1].hist(per_machine, bins=range(0, per_machine.max() + 2), color="#1D9E75", align="left")
axes[1].set(title="Failures per machine", xlabel="Failures in window", ylabel="Machines")
plt.tight_layout()
plt.show()

top10_share = per_machine.sort_values(ascending=False).head(10).sum() / per_machine.sum()
print(f"Top 10% of machines account for {top10_share:.1%} of all failures")

**Why the concentration check matters.** If failures were spread evenly, the top 10% of machines would hold about 10% of failures. A much higher share means machine level effects (age, usage, model) are real and a single fleet average will misprice contracts: good tools subsidize bad ones.

Next, the time view. A flat monthly trend supports modeling failures as a steady process; a trend or spikes would need to be explained before forecasting.

In [ ]:
monthly = (failures.set_index("datetime")
           .groupby([pd.Grouper(freq="MS"), "failure"]).size()
           .unstack(fill_value=0).reindex(columns=COMPONENTS, fill_value=0))
monthly.plot(marker="o", title="Monthly failures by component")
plt.ylabel("Failures")
plt.show()

## 6. Linking failures to maintenance records
Each failure should have a matching maintenance record (same machine, same time, same component). We verify this, then label every replacement as:

1. **Corrective:** the component failed and was replaced. The lifetime ended in an **event**.
2. **Preventive:** the component was replaced while still working. The lifetime is **censored**: we only know it would have lasted *at least* this long.

In [ ]:
fail_keys = failures.rename(columns={"failure": "comp"})[["machineID", "datetime", "comp"]]
fail_keys["is_failure"] = 1

maint = (maint.merge(fail_keys, on=["machineID", "datetime", "comp"], how="left")
              .fillna({"is_failure": 0}).astype({"is_failure": int}))

matched = maint["is_failure"].sum()
print(f"Failures: {len(failures):,}   matched to a maintenance record: {matched:,}   "
      f"unmatched: {len(failures) - matched:,}")

in_window = maint["datetime"] >= OBS_START
print("\nReplacements inside observation window:")
print(maint.loc[in_window, "is_failure"].map({1: "corrective", 0: "preventive"}).value_counts())

## 7. Building the component lifetime table
This is the core step. For each machine and component, sort replacements by time. The time between two consecutive replacements is one **lifetime** of that component.

### Why censoring matters (numerical example)
Imagine three `comp2` units:

| Unit | Lifetime (days) | How it ended |
|---|---|---|
| A | 90 | failed |
| B | 150 | preventive swap (still working) |
| C | 200 | still running at end of data |

The naive approach averages only failures: mean life = 90 days. Dropping B and C throws away the evidence that components can survive 150 and 200+ days, so the fleet looks far worse than it is. Treating B and C as failures instead gives (90 + 150 + 200) / 3 = 147 days, which is also wrong because it pretends they failed.

The right estimate uses **exposure**: 1 failure over 90 + 150 + 200 = 440 component days gives a failure rate of 1 / 440 per day, or an MTBF (Mean Time Between Failures) of 440 days under a constant rate assumption. Survival models in Part 2 generalize exactly this idea.

### Rules for each lifetime
1. **Start:** a recorded replacement (installation date is known).
2. **End:** the next replacement of the same component, or `OBS_END` if none (censored).
3. **Event:** 1 if the ending replacement was a failure, else 0.
4. **Keep only lifetimes ending inside the observation window**, since earlier endings cannot be classified.

Lifetimes before a component's *first* recorded replacement are excluded: we do not know when those parts were installed.

In [ ]:
def build_lifetimes(maint: pd.DataFrame, obs_start: pd.Timestamp, obs_end: pd.Timestamp) -> pd.DataFrame:
    df = maint.sort_values(["machineID", "comp", "datetime"]).copy()
    grp = df.groupby(["machineID", "comp"])

    df["start"] = df["datetime"]
    df["end"] = grp["datetime"].shift(-1)
    df["event"] = grp["is_failure"].shift(-1)

    # Last replacement for each machine/component: still running at end of data
    last = df["end"].isna()
    df.loc[last, "end"] = obs_end
    df.loc[last, "event"] = 0

    df["event"] = df["event"].astype(int)
    df["duration_days"] = (df["end"] - df["start"]).dt.total_seconds() / 86_400
    df["ended_by"] = np.select([df["event"] == 1, last], ["failure", "end_of_data"], "preventive")

    df = df[(df["end"] >= obs_start) & (df["duration_days"] > 0)]
    return df[["machineID", "comp", "start", "end", "duration_days", "event", "ended_by"]].reset_index(drop=True)

lifetimes = build_lifetimes(maint, OBS_START, OBS_END).merge(machines, on="machineID", how="left")
print(f"{len(lifetimes):,} component lifetimes")
lifetimes.head()

### Sanity checks
Every failure in the window should appear exactly once as an event, durations must be positive, and every lifetime needs machine attributes. These asserts stop the notebook if the table is wrong, which is far cheaper than discovering it in a cost forecast.

In [ ]:
failures_in_window = failures[failures["datetime"] >= OBS_START]
matched_in_window = maint[(maint["datetime"] >= OBS_START) & (maint["is_failure"] == 1)]

assert lifetimes["event"].sum() == len(matched_in_window), "Event count does not match failures"
assert (lifetimes["duration_days"] > 0).all(), "Non positive durations found"
assert lifetimes[["model", "age"]].notna().all().all(), "Missing machine attributes"

print(f"Events in table: {lifetimes['event'].sum():,}  |  failures in window: {len(failures_in_window):,}")
lifetimes["ended_by"].value_counts()

## 8. First look at reliability: exposure based failure rates
Before any model, compute the crude failure rate for each component exactly as in the numerical example: **failures divided by total component days of exposure**. Annualizing it gives an AFR (Annualized Failure Rate), the metric service teams already use.

$$\text{AFR} = \frac{\text{failures}}{\text{component days}} \times 365$$

This number is the baseline every later model must beat.

In [ ]:
rates = (lifetimes.groupby("comp")
         .agg(lifetimes=("event", "size"), failures=("event", "sum"), exposure_days=("duration_days", "sum")))
rates["AFR"] = rates["failures"] / rates["exposure_days"] * 365
rates["MTBF_days"] = rates["exposure_days"] / rates["failures"]
rates["censored_share"] = 1 - rates["failures"] / rates["lifetimes"]
rates.round(3)

**Reading the table:** an AFR of 0.5 means each installed unit of that component is expected to fail about 0.5 times per year. On a fleet of 100 machines that is roughly 50 replacements a year for that component alone, which is already a first draft of a parts and labor forecast.

The `censored_share` column shows how much information the naive "average of failures" approach would have thrown away.

Finally, compare how lifetimes are distributed for failures versus censored units.

In [ ]:
fig, axes = plt.subplots(1, len(COMPONENTS), figsize=(14, 3.5), sharey=True)
for ax, comp in zip(axes, COMPONENTS):
    sub = lifetimes[lifetimes["comp"] == comp]
    ax.hist(sub.loc[sub["event"] == 1, "duration_days"], bins=20, alpha=0.6, label="failed", color="#D85A30")
    ax.hist(sub.loc[sub["event"] == 0, "duration_days"], bins=20, alpha=0.6, label="censored", color="#1D9E75")
    ax.set(title=comp, xlabel="Lifetime (days)")
axes[0].set_ylabel("Lifetimes")
axes[0].legend()
plt.tight_layout()
plt.show()

## 9. Save and summarize

In [ ]:
out_path = DATA_PROCESSED / "component_lifetimes.csv"
lifetimes.to_csv(out_path, index=False)
print(f"Saved {len(lifetimes):,} rows to {out_path}")

### Summary
1. Profiled five tables and found that failures are only observable in 2015, so we defined an explicit observation window.
2. Characterized the installed base by model and age, and checked whether failures concentrate in a subset of machines.
3. Linked every failure to its maintenance record and separated corrective from preventive replacements.
4. Built a component lifetime table that keeps censored lifetimes instead of discarding them.
5. Computed exposure based AFR and MTBF per component as the baseline for later models.

### Next: Part 2, survival modeling
Using `component_lifetimes.csv` we will fit Kaplan Meier curves, estimate Weibull shape parameters to tell infant mortality from wear out, and use Cox PH regression to measure how machine age, model, and telemetry drive failure risk.